# Urban Informatics Final Project: Cleaned Data for Final Study Areas

Spring 2026

**Contributors:** Anwar Baroudi, Phoebe Chiu, Noelani Fixler, Michael Huang

In [23]:
# Import packages
import pandas as pd
import geopandas as gpd

In [24]:
# Import raw and intermediary processed data from Github
mtc_epcs = gpd.read_file('https://github.com/MiAnPh/MAPN-SRTS/raw/main/data/raw/equity_priority_communities_2020_acs2018.zip')
proj_schools_public_gdf = gpd.read_file('https://github.com/MiAnPh/MAPN-SRTS/raw/main/data/processed/proj_schools_public.geojson')
bus = pd.read_csv('https://github.com/MiAnPh/MAPN-SRTS/raw/main/data/raw/actransit_gtfs.csv')

# Filter the `mtc_epcs` dataframe for Alameda County EPCs only
mtc_ac_epcs = mtc_epcs[mtc_epcs['county_fip'] == '001']

# Create a geodataframe from the AC Transit GTFS data
bus_gdf = gpd.GeoDataFrame(
    bus,
    geometry=gpd.points_from_xy(bus['stop_lon'], bus['stop_lat'],
    crs = 4326)
    )


In [25]:
# Create an array of school types
school_types = proj_schools_public_gdf['SchoolType'].unique()

# Create a list of school types
school_types_list = school_types.tolist()

# Create a for loop to create the binary variables
for v in school_types:
    proj_schools_public_gdf[f"{v}"] = (proj_schools_public_gdf["SchoolType"] == v).astype(int)

In [27]:
# Create a dataframe with important sites attributes
site_school_info = proj_schools_public_gdf.groupby('geoid').agg(
    SchoolCount=('EnrollTotal', 'size'),
    EnrollTotal=('EnrollTotal', 'sum'),
    SEDTotal=('SEDCount', 'sum'),
    **{col: (col, 'sum') for col in school_types_list},
    tot_pop=('tot_pop', 'first'),
    pct_poc=('pct_poc', 'first'),
    pct_spfam=('pct_spfam', 'first'),
    pct_lep=('pct_lep', 'first'),
    pct_below2=('pct_below2', 'first'),
    pct_zvhhs=('pct_zvhhs', 'first')
).reset_index()

In [28]:
# Create a variable with the percentage of students
site_school_info['pct_students'] = site_school_info['EnrollTotal'] / site_school_info['tot_pop']

In [29]:
# Create a variable with the percentage of SED students
site_school_info['sed_share'] = site_school_info['SEDTotal'] / site_school_info['EnrollTotal']

In [30]:
# Create a variable with average school size
site_school_info['avg_school_size'] = site_school_info['EnrollTotal'] / site_school_info['SchoolCount']

In [31]:
# Conduct a spatial join of the bus stops with the epcs
proj_bus_stops_gdf = gpd.sjoin(bus_gdf, mtc_ac_epcs, how='inner', predicate='within')

In [32]:
# Create a dataframe with number of bus stops per Alameda County EPC
site_bus_info = proj_bus_stops_gdf.groupby('geoid').agg(
    StopCount=('stop_code', 'size')
).reset_index()

In [33]:
# Add the bus stop data to `potential_public_sites`
potential_public_sites = pd.merge(mtc_ac_epcs, site_bus_info, on='geoid', how='inner')

In [34]:
# Add the school data to `potential_public_sites`
potential_public_sites = pd.merge(potential_public_sites, site_school_info, on='geoid', how='left')

In [35]:
# Drop all columns ending with '_y'
potential_public_sites = potential_public_sites.loc[
    :, ~potential_public_sites.columns.str.endswith('_y')
]

# Rename columns to remove '_x'
potential_public_sites.columns = (
    potential_public_sites.columns.str.replace('_x$', '', regex=True)
)

In [36]:
# Create a column for bus stops per 1000 students
potential_public_sites['bus_stop_per_cap'] = potential_public_sites['StopCount'] / potential_public_sites['EnrollTotal'] * 1000

In [37]:
# Only keep EPCs with one of the following schools
filtered_public_sites = potential_public_sites[
    (potential_public_sites['K-12'] > 0) |
    (potential_public_sites['High'] > 0) |
    (potential_public_sites['Middle'] > 0)]

In [38]:
# Run summary statistics
filtered_public_sites[['SchoolCount', 'EnrollTotal', 'SEDTotal',
                        'K-12', 'High','Middle',
                        'pct_zvhhs', 'pct_students', 'sed_share',	'avg_school_size', 'bus_stop_per_cap']].describe()

,SchoolCount,EnrollTotal,SEDTotal,K-12,High,Middle,pct_zvhhs,pct_students,sed_share,avg_school_size,bus_stop_per_cap
count,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000,30.000000
mean,2.666667,1267.200000,978.633333,0.433333,0.466667,0.533333,0.214094,0.292426,0.790513,559.256111,20.206733
std,1.470007,780.042147,618.919497,0.773854,0.681445,0.571346,0.160316,0.181855,0.207598,559.600030,17.677378
min,1.000000,223.000000,176.000000,0.000000,0.000000,0.000000,0.045121,0.032347,0.237888,193.000000,1.969796
25%,1.250000,620.000000,433.750000,0.000000,0.000000,0.000000,0.100670,0.164848,0.751068,325.100000,7.134974
50%,2.000000,1083.000000,877.500000,0.000000,0.000000,0.500000,0.180429,0.266433,0.857029,440.750000,13.808391
75%,4.000000,1649.750000,1502.750000,1.000000,1.000000,1.000000,0.261532,0.396608,0.937813,550.916667,27.333115
max,6.000000,3220.000000,2249.000000,3.000000,2.000000,2.000000,0.631605,0.832013,0.982729,3220.000000,71.748879


In [39]:
# Apply filters for filtered dataframe
final_sites_1 = filtered_public_sites[
    (filtered_public_sites['pct_students'] > 0.25) &
    (filtered_public_sites['pct_zvhhs'] > 0.15)][['geoid', 'SchoolCount', 'EnrollTotal', 'SEDTotal',
                        'K-12', 'High','Middle',
                        'pct_zvhhs', 'pct_students', 'sed_share',	'avg_school_size', 'bus_stop_per_cap']]

In [40]:
# Apply filters for filtered dataframe
final_sites_2 = filtered_public_sites[
    (filtered_public_sites['bus_stop_per_cap'] < 12) &
    (filtered_public_sites['pct_zvhhs'] > 0.15)][['geoid','SchoolCount', 'EnrollTotal', 'SEDTotal',
                        'K-12', 'High','Middle',
                        'pct_zvhhs', 'pct_students', 'sed_share',	'avg_school_size', 'bus_stop_per_cap']]

In [41]:
# Get final_sites
final_sites = pd.concat([final_sites_1, final_sites_2])

# Create a list
final_sites = final_sites['geoid'].unique().tolist()

# Update final_sites as a dataframe
final_sites = filtered_public_sites[filtered_public_sites['geoid'].isin(final_sites)]

In [42]:
# Explore the project dataframe of schools
m = proj_schools_public_gdf.explore(color="blue", name="Schools")

final_sites.explore(
    m=m,
    color="red",
    name="Study Areas"
)
m

In [43]:
# Finalize sites
final_sites = final_sites[
    (final_sites['geoid'] == '06001402500') | # West Oakland
    (final_sites['geoid'] == '06001406000') | # Nimitz
    (final_sites['geoid'] == '06001408800') # Airport
     ]

# Create a dataframe for each individual EPC
wo_epc_gdf = final_sites[final_sites['tract'] == 402500]
nimitz_epc_gdf = final_sites[final_sites['tract'] == 406000]
oak_airport_epc_gdf = final_sites[final_sites['tract'] == 408800]

# Get a truncated dataframe
final_sites[['geoid', 'tot_pop', 'pct_zvhhs',  'epc_class', 'StopCount',
       'SchoolCount', 'EnrollTotal', 'SEDTotal', 'K-12', 'Elementary',
     'High', 'Middle',
       'Continuation', 'pct_students', 'sed_share',
       'avg_school_size', 'bus_stop_per_cap']].T

,8,25,47
geoid,06001402500,06001406000,06001408800
tot_pop,1786,3344,7054
pct_zvhhs,0.539548,0.234812,0.316194
epc_class,Highest,Higher,Highest
StopCount,14,21,25
SchoolCount,3.0,5.0,5.0
EnrollTotal,579.0,1036.0,2125.0
SEDTotal,569.0,888.0,2076.0
K-12,0.0,0.0,2.0
Elementary,1.0,2.0,1.0


In [44]:
# Correct the variable type of the `tract` column of the sschools dataframe
proj_schools_public_gdf['geoid'] = proj_schools_public_gdf['geoid'].astype(float).astype(int)

# Filter the schools dataframe to only include our EPCs of interest
final_schools_gdf = proj_schools_public_gdf[
    (proj_schools_public_gdf['geoid'] == 6001402500) | # West Oakland
    (proj_schools_public_gdf['geoid'] == 6001406000) | # Nimitz
    (proj_schools_public_gdf['geoid'] == 6001408800) # Airport
]

# Drop elementary schools in the final schools dataframe
final_schools_gdf = final_schools_gdf[final_schools_gdf['GradeHigh'] != '05']

# Create individual dataframes for the schools
wo_epc_schools = final_schools_gdf[final_schools_gdf['geoid'] == 6001402500]
nimitz_epc_schools = final_schools_gdf[final_schools_gdf['geoid'] == 6001406000]
oak_airport_epc_schools = final_schools_gdf[final_schools_gdf['geoid'] == 6001408800]

In [45]:
# Create the final bus stops
final_bus_stops_gdf = gpd.sjoin(bus_gdf, final_sites, how='inner', predicate='within')

# Create individual dataframes for the bus stops
wo_bus_stops = final_bus_stops_gdf[final_bus_stops_gdf['geoid'] == '06001402500']
nimitz_bus_stops = final_bus_stops_gdf[final_bus_stops_gdf['geoid'] == '06001406000']
oak_bus_stops = final_bus_stops_gdf[final_bus_stops_gdf['geoid'] == '06001408800']

In [46]:
# Explore final dataframes in a map

m = final_sites.explore(
    color="red",
    style_kwds={
        "fillOpacity": 0.1,
        "weight": 1
    },
    tooltip=[
        'geoid', 'SchoolCount', 'EnrollTotal', 'SEDTotal',
        'K-12', 'High', 'Middle',
        'pct_zvhhs', 'pct_students', 'sed_share',
        'avg_school_size', 'bus_stop_per_cap'
    ],
    highlight=False,
    tiles="CartoDB positron",
    name="Study Areas"
)

# Schools on top
m = final_schools_gdf.explore(
    m=m,
    color="blue",
    name="Schools",
    tooltip=["SchoolName"],
    popup=True
)

# Bus stops on top
m = final_bus_stops_gdf.explore(
    m=m,
    color="darkgreen",
    name="Bus Stops",
    tooltip=["stop_name"],
    popup=True
)

m

In [47]:
# Export the cleaned GeoDataFrames
outputs = [
    # Study Areas
    (final_sites, "../data/processed/final_epcs.geojson"),
    (wo_epc_gdf, "../data/processed/west_oakland_epc.geojson"),
    (nimitz_epc_gdf, "../data/processed/nimitz_epc.geojson"),
    (oak_airport_epc_gdf, "../data/processed/oak_airport_epc.geojson"),

    # Schools
    (final_schools_gdf, "../data/processed/final_schools.geojson"),
    (wo_epc_schools, "../data/processed/west_oakland_schools.geojson"),
    (nimitz_epc_schools, "../data/processed/nimitz_chools.geojson"),
    (oak_airport_epc_schools, "../data/processed/oak_airport_schools.geojson"),

    # Bus Stops
    (final_bus_stops_gdf, "../data/processed/final_bus_stops.geojson"),
    (wo_bus_stops, "../data/processed/west_oakland_bus_stops.geojson"),
    (nimitz_bus_stops, "../data/processed/nimitz_bus_stops.geojson"),
    (oak_bus_stops, "../data/processed/oak_airport_bus_stops.geojson"),
]

for gdf, filename in outputs:
    gdf.to_file(filename, driver="GeoJSON")